# Foreign Aid & Development — Nigeria vs Kenya

Data source: **World Bank** — Indicators API v2 (open access, no key required)  

- **Top-left**: Net ODA Received (foreign aid, billions USD)  
- **Top-right**: Labour Force Participation Rate (% of population 15+)  
- **Bottom-left**: GDP (current USD, billions)  
- **Bottom-right**: Economic Structure (Agriculture / Industry / Services as % of GDP)  

Time range: 2000 – 2023

In [1]:
import requests, time
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# ══════════════════════════════════════════════════════════════════════
# Style constants  (warm beige — matching other notebooks)
# ══════════════════════════════════════════════════════════════════════
STYLE = dict(
    background = '#f5efe2',
    text       = '#2f2a25',
    grid       = '#d8cfbf',
    legend_bg  = 'rgba(245,239,226,0.85)',
    font       = 'Georgia',
)

COUNTRIES = {
    'NGA': {'name': 'Nigeria', 'colors': {'main': '#4F7F84', 'male': '#2A9D8F', 'female': '#E76F51',
            'agri': '#8FBC8F', 'industry': '#4F7F84', 'services': '#C4787A'}},
    'KEN': {'name': 'Kenya',   'colors': {'main': '#C4787A', 'male': '#E9C46A', 'female': '#9B7CB8',
            'agri': '#D4A574', 'industry': '#494C30', 'services': '#B8860B'}},
}

DATE_RANGE = '2000:2023'
ISO_CODES  = ';'.join(COUNTRIES.keys())   # NGA;KEN


# ══════════════════════════════════════════════════════════════════════
# Helper: fetch a World Bank indicator (with retries)
# ══════════════════════════════════════════════════════════════════════
def wb_fetch(indicator: str, value_col: str, retries: int = 3) -> pd.DataFrame:
    url = f'https://api.worldbank.org/v2/country/{ISO_CODES}/indicator/{indicator}'
    rows = []
    page = 1
    while True:
        for attempt in range(retries):
            try:
                r = requests.get(url, params={'format': 'json', 'per_page': 500,
                                              'date': DATE_RANGE, 'page': page}, timeout=90)
                r.raise_for_status()
                break
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                if attempt == retries - 1:
                    raise
                time.sleep(2 ** attempt)
        data = r.json()
        if len(data) < 2 or data[1] is None:
            break
        for rec in data[1]:
            if rec['value'] is not None:
                rows.append({'iso3': rec['countryiso3code'],
                             'country': rec['country']['value'],
                             'year': int(rec['date']),
                             value_col: rec['value']})
        if page >= data[0]['pages']:
            break
        page += 1
    df = pd.DataFrame(rows).sort_values(['iso3', 'year'])
    print(f'  ✓ {indicator:30s}  →  {len(df)} rows')
    return df


# ══════════════════════════════════════════════════════════════════════
# Fetch all indicators
# ══════════════════════════════════════════════════════════════════════
print('Fetching data from World Bank …')

df_oda      = wb_fetch('DT.ODA.ODAT.CD',       'oda')            # Net ODA (US$)
df_lfp      = wb_fetch('SL.TLF.CACT.ZS',       'lfp_total')     # Labour force participation, total
df_lfp_m    = wb_fetch('SL.TLF.CACT.MA.ZS',    'lfp_male')      # Male
df_lfp_f    = wb_fetch('SL.TLF.CACT.FE.ZS',    'lfp_female')    # Female
df_gdp      = wb_fetch('NY.GDP.MKTP.CD',        'gdp')           # GDP (current US$)
df_gdp_pc   = wb_fetch('NY.GDP.PCAP.CD',        'gdp_pc')        # GDP per capita
df_agri     = wb_fetch('NV.AGR.TOTL.ZS',        'agri_pct')      # Agriculture % of GDP
df_ind      = wb_fetch('NV.IND.TOTL.ZS',        'industry_pct')  # Industry % of GDP
df_srv      = wb_fetch('NV.SRV.TOTL.ZS',        'services_pct')  # Services % of GDP

# Convert ODA & GDP to billions
df_oda['oda_b'] = df_oda['oda'] / 1e9
df_gdp['gdp_b'] = df_gdp['gdp'] / 1e9

# Merge labour-force frames
df_lf = (df_lfp.merge(df_lfp_m[['iso3','year','lfp_male']], on=['iso3','year'], how='outer')
               .merge(df_lfp_f[['iso3','year','lfp_female']], on=['iso3','year'], how='outer'))

# Merge sectors
df_sectors = (df_agri.merge(df_ind[['iso3','year','industry_pct']], on=['iso3','year'], how='outer')
                     .merge(df_srv[['iso3','year','services_pct']], on=['iso3','year'], how='outer'))

print('\nAll data fetched ✓')


# ══════════════════════════════════════════════════════════════════════
# Build 2×2 subplot figure
# ══════════════════════════════════════════════════════════════════════
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Net Foreign Aid Received (ODA)',
        'Labour Force Participation Rate',
        'GDP (current USD)',
        'Economic Structure (% of GDP)',
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

# ── 1) Foreign Aid ────────────────────────────────────────────────────
for iso3, meta in COUNTRIES.items():
    sub = df_oda[df_oda.iso3 == iso3]
    fig.add_trace(go.Scatter(
        x=sub['year'], y=sub['oda_b'],
        mode='lines+markers', name=meta['name'],
        line=dict(color=meta['colors']['main'], width=2),
        marker=dict(size=4),
        hovertemplate='<b>%{fullData.name}</b><br>Year: %{x}<br>ODA: $%{y:.2f}B<extra></extra>',
        legendgroup=meta['name'], showlegend=True,
    ), row=1, col=1)

# ── 2) Labour Force Participation ─────────────────────────────────────
for iso3, meta in COUNTRIES.items():
    sub = df_lf[df_lf.iso3 == iso3]
    # Total
    fig.add_trace(go.Scatter(
        x=sub['year'], y=sub['lfp_total'],
        mode='lines+markers', name=f"{meta['name']} Total",
        line=dict(color=meta['colors']['main'], width=2),
        marker=dict(size=4),
        hovertemplate='<b>%{fullData.name}</b><br>%{x}: %{y:.1f}%<extra></extra>',
        legendgroup=meta['name'], showlegend=False,
    ), row=1, col=2)
    # Male
    fig.add_trace(go.Scatter(
        x=sub['year'], y=sub['lfp_male'],
        mode='lines+markers', name=f"{meta['name']} Male",
        line=dict(color=meta['colors']['male'], width=1.5, dash='dash'),
        marker=dict(size=3),
        hovertemplate='<b>%{fullData.name}</b><br>%{x}: %{y:.1f}%<extra></extra>',
        legendgroup=meta['name'], showlegend=False,
    ), row=1, col=2)
    # Female
    fig.add_trace(go.Scatter(
        x=sub['year'], y=sub['lfp_female'],
        mode='lines+markers', name=f"{meta['name']} Female",
        line=dict(color=meta['colors']['female'], width=1.5, dash='dot'),
        marker=dict(size=3),
        hovertemplate='<b>%{fullData.name}</b><br>%{x}: %{y:.1f}%<extra></extra>',
        legendgroup=meta['name'], showlegend=False,
    ), row=1, col=2)

# ── 3) GDP ────────────────────────────────────────────────────────────
for iso3, meta in COUNTRIES.items():
    sub = df_gdp[df_gdp.iso3 == iso3]
    fig.add_trace(go.Scatter(
        x=sub['year'], y=sub['gdp_b'],
        mode='lines+markers', name=meta['name'],
        line=dict(color=meta['colors']['main'], width=2),
        marker=dict(size=4),
        hovertemplate='<b>%{fullData.name}</b><br>Year: %{x}<br>GDP: $%{y:.1f}B<extra></extra>',
        legendgroup=meta['name'], showlegend=False,
    ), row=2, col=1)

# ── 4) Economic Structure ─────────────────────────────────────────────
sector_labels = [('agri_pct', 'Agriculture', 'agri'),
                 ('industry_pct', 'Industry', 'industry'),
                 ('services_pct', 'Services', 'services')]

for iso3, meta in COUNTRIES.items():
    sub = df_sectors[df_sectors.iso3 == iso3]
    for col, label, ckey in sector_labels:
        fig.add_trace(go.Scatter(
            x=sub['year'], y=sub[col],
            mode='lines+markers',
            name=f"{meta['name']} {label}",
            line=dict(color=meta['colors'][ckey], width=2,
                      dash='solid' if iso3 == 'NGA' else 'dash'),
            marker=dict(size=4 if iso3 == 'NGA' else 3,
                        symbol='circle' if iso3 == 'NGA' else 'diamond'),
            hovertemplate='<b>%{fullData.name}</b><br>%{x}: %{y:.1f}%<extra></extra>',
            legendgroup=f'{label}',
            showlegend=(iso3 == 'NGA'),  # show legend once per sector
        ), row=2, col=2)


# ══════════════════════════════════════════════════════════════════════
# Layout
# ══════════════════════════════════════════════════════════════════════
fig.update_layout(
    height=850, width=1100,
    paper_bgcolor=STYLE['background'],
    plot_bgcolor=STYLE['background'],
    font=dict(color=STYLE['text'], family=STYLE['font']),
    title=dict(
        text='<b>Foreign Aid & Development — Nigeria vs Kenya</b>'
             '<br><span style="font-size:12px; color:#7f7a75">'
             'Source: World Bank  ·  2000 – 2023</span>',
        font=dict(size=22, family=STYLE['font']),
        x=0.5, xanchor='center',
    ),
    legend=dict(
        bgcolor=STYLE['legend_bg'],
        font=dict(size=11),
        x=1.02, y=1.0, xanchor='left', yanchor='top',
    ),
    hovermode='x unified',
    margin=dict(l=70, r=170, t=100, b=50),
)

# Axis styling
axis_style = dict(
    gridcolor=STYLE['grid'],
    zeroline=False,
)
fig.update_xaxes(**axis_style, dtick=4)
fig.update_yaxes(**axis_style)

# Y-axis labels
fig.update_yaxes(title_text='Billions USD',  row=1, col=1)
fig.update_yaxes(title_text='% of pop. 15+',  row=1, col=2)
fig.update_yaxes(title_text='Billions USD',  row=2, col=1)
fig.update_yaxes(title_text='% of GDP',       row=2, col=2)

# Subplot title styling
for ann in fig.layout.annotations:
    ann.font = dict(color=STYLE['text'], size=14, family=STYLE['font'])

# ── Add text annotations for labour-force subplot legend ──────────
fig.add_annotation(
    text='── solid = Total  ╌╌ dash = Male  ··· dot = Female',
    xref='x2', yref='y2',
    x=2001, y=48, showarrow=False,
    font=dict(size=9, color='#7f7a75', family=STYLE['font']),
)

# ── Add text annotation for sector subplot legend ─────────────────
fig.add_annotation(
    text='── solid ● = Nigeria  ╌╌ dash ◆ = Kenya',
    xref='x4', yref='y4',
    x=2001, y=18, showarrow=False,
    font=dict(size=9, color='#7f7a75', family=STYLE['font']),
)

# ── Export ─────────────────────────────────────────────────────────
os.makedirs('output', exist_ok=True)
fig.write_html('output/foreign_aid.html')
fig.write_image('output/foreign_aid.png', scale=2)
print('Saved → output/foreign_aid.html  &  output/foreign_aid.png')

fig.show()

Fetching data from World Bank …
  ✓ DT.ODA.ODAT.CD                  →  48 rows
  ✓ SL.TLF.CACT.ZS                  →  48 rows
  ✓ SL.TLF.CACT.MA.ZS               →  48 rows
  ✓ SL.TLF.CACT.FE.ZS               →  48 rows
  ✓ NY.GDP.MKTP.CD                  →  48 rows
  ✓ NY.GDP.PCAP.CD                  →  48 rows
  ✓ NV.AGR.TOTL.ZS                  →  48 rows
  ✓ NV.IND.TOTL.ZS                  →  48 rows
  ✓ NV.SRV.TOTL.ZS                  →  42 rows

All data fetched ✓
Saved → output/foreign_aid.html  &  output/foreign_aid.png
